# Setup

In [10]:
# Standard library imports
import time
import warnings

# Third-party imports (core data -> visualization)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb            # gradient-boosted trees, used in the real-data section

# Configuration & Settings
warnings.simplefilter(action='ignore', category=FutureWarning)

What we import:
- **`time`** is the standard-library stopwatch 
- **`warnings`** lets us silence noisy-but-harmless `FutureWarning` messages from the numerical libraries so the output stays readable (mostly for pandas df manipulations); `simplefilter(action='ignore', ...)` installs that filter globally.
- **`numpy`**  is the numerical engine: arrays, random number generation, and the least-squares solver 
- **`pandas`** gives us the `DataFrame`, a labelled table we use only at the end to assemble a tidy comparison of results. 
- **`matplotlib.pyplot`**  is the plotting library; every figure in this notebook goes through it. 
- **`xgboost`** is the gradient-boosting library

Important: none of these are conformal-prediction-specific - CP needs no special distributional machinery, it sits on top of whatever forecaster you already have.

In [11]:
# general settings
class CFG:
    SEED = 42                 # master random seed for reproducibility
    ALPHA = 0.10              # target miscoverage -> 1 - ALPHA = 90% nominal coverage
    P_LAGS = 2                # number of autoregressive lags in the covariate
    N_TRAIN = 300             # points used to FIT the base forecaster
    N_CAL = 300               # points used to CALIBRATE the conformal quantile
    N_TEST = 300              # points used to EVALUATE coverage and width
    N_REPS = 50               # Monte-Carlo repetitions in the simulation study
    img_dim1 = 13             # default figure width
    img_dim2 = 5              # default figure height

# display style
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (CFG.img_dim1, CFG.img_dim2)

np.random.seed(CFG.SEED)

The **`CFG`** class is a single namespace holding every constant the notebook depends on, so there are no unexplained "magic numbers" scattered through the code:
-  **`SEED = 42`** seeds NumPy's global generator; fixing it means every run reproduces the same data and the same numbers 
- **`ALPHA = 0.10`** is the *miscoverage* level - we tolerate being wrong 10% of the time
- **`P_LAGS = 2`** says our forecaster predicts each value from its two previous values. 
- The trio **`N_TRAIN / N_CAL / N_TEST = 300 / 300 / 300`** is the heart of split conformal prediction: one block to *fit* the model, a disjoint block to *calibrate* the error threshold, and a final block to *test*. Keeping these separate and **chronological** (train is the oldest, test is the newest) is what makes the evaluation simulate real-life forecasting on unseen future data.
-  **`N_REPS = 50`** is how many independent series we average over later to get stable estimates.

`plt.style.use("seaborn-v0_8")` applies a clean default theme, `rcParams["figure.figsize"]` sets a wide-and-short default so time series are legible. 

The final `np.random.seed` call locks in reproducibility before any data is generated (insofar as possible in Python ML ;-)

# Utils

In [12]:
def conformal_quantile(scores, alpha):
    '''Finite-sample (1 - alpha) conformal quantile of a set of nonconformity scores.'''
    s = np.sort(scores)
    n = len(s)
    k = int(np.ceil((1 - alpha) * (n + 1)))   # finite-sample rank correction
    if k > n:
        return np.inf                          # not enough points -> unbounded interval
    return s[max(k, 1) - 1]

Most important function in conformal prediction:
- Given a 1-D array of *nonconformity scores* (absolute residuals `|y - prediction|` measured on the calibration set) and a miscoverage level `alpha`, it returns the threshold `q` such that roughly `1 - alpha` of the scores fall at or below it. We sort the scores, then pick the element at rank `k = ceil((1 - alpha)(n + 1))`.
- A "naive" empirical quantile would take the `(1 - alpha)`-th fraction of `n` points. The conformal correction inflates this slightly: it reserves a "slot" for the unseen test point, treating the calibration scores *and* the future score as one exchangeable pool of `n + 1` values. This adjustment upgrades an asymptotic statement into a **finite-sample guarantee**: with this `k`, the interval `[prediction - q, prediction + q]` provably covers the truth with probability at least `1 - alpha` whenever the data are exchangeable.
- When `1 - alpha` is so high (or `n` so small) that `k` exceeds `n`, no observed score is large enough to promise the coverage => **infinite interval** . Returning `np.inf` makes that explicit.

In [13]:
def weighted_conformal_quantile(scores, weights, test_weight, alpha):
    '''Weighted (1 - alpha) conformal quantile. Calibration points carry unequal weight.'''
    s = np.append(scores, np.inf)                 # the unseen test score sits at +inf
    w = np.append(weights, test_weight).astype(float)
    w = w / w.sum()                               # normalise weights to sum to 1
    order = np.argsort(s)
    s, w = s[order], w[order]
    cum = np.cumsum(w)
    idx = min(np.searchsorted(cum, 1 - alpha, side="left"), len(s) - 1)
    return s[idx]

This function generalises the quantile to the case where calibration points are **not** equally trustworthy:
- Each calibration score `s_i` gets a non-negative weight `w_i`. We append the test point as a score of `+inf` with its own `test_weight`, normalise all weights to sum to one, then walk up the sorted scores accumulating weight until we first reach the `1 - alpha` mass. That score is the weighted quantile.
- The reason we include `+inf` test slot is that it accounts for the unknown future point inside the weighted pool. If the calibration weights are too concentrated on too few points, the cumulative mass can only cross `1 - alpha` at the `+inf` slot => yields an unbounded interval.
- With all weights equal, this collapses back to ordinary conformal prediction. If more recent points are weighted higher, the quantile is driven by *recent* errors.

In [14]:
def interval_metrics(y, lower, upper, alpha):
    '''Coverage, mean width, and the Winkler interval score for a batch of intervals.'''
    covered = (y >= lower) & (y <= upper)
    coverage = float(np.mean(covered))
    width = float(np.mean(upper - lower))
    penalty = np.where(y < lower, (2 / alpha) * (lower - y),
              np.where(y > upper, (2 / alpha) * (y - upper), 0.0))
    winkler = float(np.mean((upper - lower) + penalty))
    return {"coverage": round(coverage, 3),
            "width": round(width, 3),
            "winkler": round(winkler, 3)}

This function is a scorecard, allowing to balance validity and efficiency (you can always cover everything with an infinitely wide interval  ;-)
- **Coverage** is the empirical fraction of test points that land inside their interval. A valid 90% method should sit near `0.90`
- **Width** is the mean interval length: among methods that *are* valid, the one with the smaller width is sharper and more useful. 
- **Winkler score** (the interval score) is the tie-breaker that combines both: it charges you the width of every interval *plus* a penalty of `(2/alpha) * distance` whenever the truth escapes the interval. It rewards intervals that are tight *and* honest, and punishes the cheap trick of inflating width to buy coverage. **Lower Winkler is better.** We round to three decimals and return a dict so results slot straight into a table.

In [15]:
def chronological_split(X, y, n_train, n_cal, n_test):
    '''Split lagged design matrix into train / calibration / test blocks, in time order.'''
    tr = slice(0, n_train)
    ca = slice(n_train, n_train + n_cal)
    te = slice(n_train + n_cal, n_train + n_cal + n_test)
    return (X[tr], y[tr]), (X[ca], y[ca]), (X[te], y[te])

A one-line helper for a very fundamental rule: **splits are chronological, never random**. 
The oldest `n_train` rows fit the model, the `n_cal` calibrate the quantile, and the final `n_test` are held out for evaluation. The function returns three `(X, y)` tuples carved out with `slice` objects.

In [16]:
def plot_intervals(y, lower, upper, point=None, title="", ax=None):
    '''Overlay observed series, the prediction band, and any points the band misses.'''
    if ax is None:
        fig, ax = plt.subplots()
    t = np.arange(len(y))
    ax.plot(t, y, color="C0", lw=2, label="observed")
    if point is not None:
        ax.plot(t, point, color="red", lw=1, alpha=0.7, label="point forecast")
    ax.fill_between(t, lower, upper, color="green", alpha=0.20, label="90% interval")
    miss = (y < lower) | (y > upper)
    ax.scatter(t[miss], y[miss], color="red", s=18, zorder=3, label="missed")
    cov = np.mean(~miss)
    ax.set_title(f"{title}  (coverage = {cov:.2f})")
    ax.legend(loc="upper left", fontsize=8)
    return ax

The plotting wrapper we reuse for every method: it draws the **observed series** as a solid blue line (`lw=2`), (optionally) the **point forecast** as a thin red line, shades the **90% interval** as a translucent green band, and marks every **missed** point ( where the truth fell outside the band) as a red scatter sitting on top (`zorder=3`).

 The realised coverage is printed in the title so you can read validity at a glance. 

# Groundwork

In [17]:
def simulate_dgp(name, n, rng, shift=1.0):
    '''Generate one realisation of a named data-generating process of length n.'''
    e = rng.normal(0, 1, size=n)         # i.i.d. standard-normal innovations
    y = np.zeros(n)
    if name == "AR(1)":                  # weak temporal dependence, stationary
        for t in range(1, n):
            y[t] = 0.8 * y[t - 1] + e[t]
    elif name == "ARMA(1,1)":            # slightly longer memory, stationary
        for t in range(1, n):
            y[t] = 0.5 * y[t - 1] + e[t] + 0.4 * e[t - 1]
    elif name == "Mean-Shift":           # abrupt, permanent distribution shift
        t_star = (2 * n) // 3            # shift lands exactly at the first test point
        mu = np.where(np.arange(n) >= t_star, shift, 0.0)
        y = mu + e
    elif name == "GARCH(1,1)":           # volatility clustering (heteroscedastic)
        for t in range(1, n):
            sigma2 = 0.4 + 0.5 * y[t - 1] ** 2
            y[t] = np.sqrt(sigma2) * e[t]
    else:
        raise ValueError(f"unknown process: {name}")
    return y

These four processes are chosen to probe the two ways time series break exchangeability.

The first two are **stationary and weakly dependent** — the "easy" regime where standard CP should still work. **AR(1)**, `y_t = 0.8 y_{t-1} + e_t`, has simple autocorrelation: each value is a damped echo of the last plus fresh noise. **ARMA(1,1)** adds a moving-average term `0.4 e_{t-1}`, giving slightly richer memory. Both forget their past quickly (they are *β-mixing*), so the temporal dependence is mild.

The last two are the stress tests. **Mean-Shift** is pure noise around a level that **jumps permanently** by `shift = 1.0` two-thirds of the way through — engineered so the break coincides with the very first test point. This is *distribution shift*: the calibration block and the test block come from genuinely different processes, and no reweighting of old data can know about it. **GARCH(1,1)** keeps a constant mean but its *variance* clusters — `sigma2 = 0.4 + 0.5 y_{t-1}^2` means a big move begets a big move — so calm and turbulent regimes alternate. A single fixed interval width cannot be right in both.

Mechanically, every process is driven by the same i.i.d. innovations `e`; the `name` selects how those innovations are wired into a series. Passing an explicit `rng` (a NumPy random generator) keeps each realisation reproducible.